# 05 — Evaluation, Error Analysis & Transfer

**Part of the 5-notebook split** of `code-mixed-hate-speech-detection-pipeline-v2.ipynb`.
The complete original notebook is kept unchanged for upload.

**Scope:** All remaining sections in original order: evaluation (5.1–5.8), error analysis (6.1–6.4), MeHate cross-lingual transfer (7.1–7.3), self-tests (8), wrap-up (9).

**Prerequisites:** 01–04 (processed splits + split_state accumulators + seed-42 model weights).

**Writes:** All result CSVs, remaining figures, transfer.csv, run_manifest.json, requirements-freeze.txt

> Run 01 → 05 in order in the same environment: Kaggle with **Internet ON** and
> **GPU T4 x2**. Every assigned code cell is copied verbatim from the original;
> only the small handoff cells are new. Outputs are intentionally cleared — the
> executed results live in the unchanged original notebook, the PDF, and the
> `results/` artifacts.

### 0.1 Dependencies

In [ ]:
import sys, subprocess

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

# transformers>=4.46 required: eval_strategy + processing_class were introduced in 4.46.
pip_install([
    "transformers>=4.46,<5",
    "datasets>=3.0",
    "accelerate>=0.34",
    "scipy>=1.11",
])

### 0.2 Imports & Global Configuration (single source of truth for the environment)

In [ ]:
import os, re, json, math, time, random, hashlib, shutil, warnings, urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

import torch
from torch.utils.data import Dataset as TorchDataset

from datasets import load_dataset
from huggingface_hub import HfApi
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer, set_seed as hf_set_seed,
)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    f1_score, precision_recall_fscore_support, matthews_corrcoef,
    balanced_accuracy_score, confusion_matrix, classification_report,
)
from scipy import stats

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 110

# ---- Environment contract: the ONLY place run mode is decided ---------------
SMOKE_TEST = False   # True = tiny data + tiny model, CPU fast pass (~2 min)

PRISM_ID  = "pankajbiswas6/prism-hinglish-hate-speech"
MEHATE_REPO = "l3cube-pune/MarathiNLP"          # GitHub repo, CSV fetched at HEAD SHA
MEHATE_CSV  = "MeEval/L3Cube-MeHate/codemix-hate-small-test.csv"

SEEDS = [42, 43, 44]
TINY_MODEL = "sshleifer/tiny-distilroberta-base"
MODELS = {
    "hing-roberta": "l3cube-pune/hing-roberta",
    "muril":        "google/muril-base-cased",
    "xlm-roberta":  "xlm-roberta-base",
}
if SMOKE_TEST:
    MODELS = {k: TINY_MODEL for k in MODELS}

MAX_LEN     = 128 if not SMOKE_TEST else 32
NUM_EPOCHS  = 3   if not SMOKE_TEST else 1
BATCH_SIZE  = 32  if not SMOKE_TEST else 4
BOOTSTRAP_N = 1_000 if not SMOKE_TEST else 100
SMOKE_N     = 200
LR, WEIGHT_DECAY, WARMUP_RATIO = 2e-5, 0.01, 0.1   # named, not magic

WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("./kaggle_working")
RESULTS_DIR, FIGURES_DIR, PREDS_DIR = WORK_DIR/"results", WORK_DIR/"figures", WORK_DIR/"preds"
for d in (RESULTS_DIR, FIGURES_DIR, PREDS_DIR):
    d.mkdir(parents=True, exist_ok=True)

# Fail fast: the full run is impossible on CPU and would silently drop fp16.
if not SMOKE_TEST:
    assert torch.cuda.is_available(), (
        "GPU required: set Accelerator to 'GPU T4 x2' (or P100) in notebook settings. "
        "For a CPU pass, set SMOKE_TEST = True in section 0.2.")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"SMOKE_TEST = {SMOKE_TEST} | Device = {DEVICE} | Work dir = {WORK_DIR.resolve()}")

### 0.3 Reproducibility helpers

In [ ]:
def set_seed(seed: int) -> None:
    """Seed python, numpy and torch (incl. CUDA). Note: with fp16 + cuDNN this is
    reproducible-in-practice, not bit-exact; `full_determinism=True` would cost speed."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

ALL_SEED_RESULTS = []   # one row per (model, seed, split)
PER_CLASS_RESULTS = []  # per-class P/R/F1 rows
COSTS = []              # params / train time / ms-per-item per (model, seed)
THRESH_CURVES = {}      # (model_key, seed) -> (thresholds, val macro-F1 curve)
LOG_HISTORY = {}        # (model_key, seed) -> trainer.state.log_history
TRAINED = {}            # (model_key, seed=SEEDS[0]) -> artifacts needed in sections 6-7

print("Reproducibility helpers ready.")

In [ ]:
# ---- Shared handoff directory for the 5-part notebook chain ------------------
# Run 01 -> 02 -> 03 -> 04 -> 05 in the SAME environment (Kaggle, Internet ON, GPU).
STATE_DIR = WORK_DIR / "split_state"
STATE_DIR.mkdir(parents=True, exist_ok=True)
print("Split state dir:", STATE_DIR)

### Shared helpers (verbatim definitions from the complete notebook)

Only the definitions needed by this part are copied here — no duplicate section
output. Original logic is unchanged.

In [ ]:
### Data & preprocessing helpers (verbatim definitions)

def pick_split(names, wanted):
    # Map a canonical name (train/val/test) onto whatever the repo calls it.
    aliases = {"train": ["train"], "val": ["val", "validation", "dev", "valid"], "test": ["test"]}[wanted]
    for a in aliases:
        if a in names:
            return a
    raise ValueError(f"No split matching {wanted!r} in {names}")

URL_RE    = re.compile(r"https?://\S+|www\.\S+")

EMAIL_RE  = re.compile(r"[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}")

PHONE_RE  = re.compile(r"(?:\+?\d[\d\s\-()]{7,}\d)")

USER_RE   = re.compile(r"@\w+")

LONG_RUN  = re.compile(r"([A-Za-z\u0900-\u097F])\1{3,}")

def deidentify(text: str) -> str:
    t = URL_RE.sub("<URL>", text)
    t = EMAIL_RE.sub("<EMAIL>", t)
    t = PHONE_RE.sub("<PHONE>", t)
    t = USER_RE.sub("<USER>", t)
    t = LONG_RUN.sub(r"\1\1", t)
    return t.strip()

DEVANAGARI = re.compile(r"[\u0900-\u097F]")

TOKEN_RE   = re.compile(r"[A-Za-z\u0900-\u097F]+")

PLACEHOLDER_RE = re.compile(r"<URL>|<USER>|<EMAIL>|<PHONE>")

ROMAN_HINDI = set("""
hai hain he hn nahi nahin nai nhi kya kyu kyun ku aue aur ya yaani mein me mai maiya ka ki ke ko se
par per bhi bi hi to tho toh tu tum tumhari tumhara tumhare hum humara humari apna apni apne aap
yeh ye yehe woh vo wo voh kaha kahan kab kaisa kaise kaisi itna itni jaisa jaise abhi ab kal aaj
log logon logo logo sabh sab sabhi kuch koi bahut bohut bohot bhut thoda thodi zyada jyada kam
acha achha acha achhi accha theek thik karo karna karte kar kiya kiye kardia diya diye liya liye
lena dena jata jaate jana gaya gaye gayi raha rahe rahi hogaya hogae hojayega mila mile milega
wala wali waale vale vala walla banda bande log insan insaan baat baatein baat bat kaam kaam ka
desh deshdrohi watan hindi hindustan urdu muslim hindu sikh christian pakistani bangladeshi
bharat bharatiya mullah mulla maulvi katuwa katua katwe jihadi jihad maza hab IB dat
paisa paise rupaye free bijli sadak paani school hospital naukri berojgari mehangai
gali galia gaali gaaliya galiyan maader mader behen behanchod bhosdi bhosdike bhosda chutiya
chutiye chutiye madarchod behenchod randi harami haramkhor kamina kamine kutta kutte kamina
pagal paagal bewakoof bewakoofon bakwaas bakwas bakwaas faltu ghatiya nich neech jhoota jhooth
chor chchor gaddar dhokha betray saala saale sala saley hijre hijra chakka anpadh gawar
bhookha mara maar marey marenge pitai peet pit jhagda ladai fight phir fir jab tab kabhi
kyunki kyunke liye waha yahan idhar udhar aisa waisa kaisa hoga hogi honge thi the thin
na haan han hn ji nahiin chal chalo aaja aao dekh dekho sun suno ruk jaldi abhi turant
""".split())

def _lang_of(tok: str) -> str:
    if DEVANAGARI.search(tok):
        return "hi"
    return "hi" if tok.lower() in ROMAN_HINDI else "en"

def code_mixing_stats(texts):
    rows = []
    for t in texts:
        cleaned = PLACEHOLDER_RE.sub(" ", t)
        toks = TOKEN_RE.findall(cleaned)
        n = len(toks)
        chars = re.sub(r"\s+", "", t)
        devi = len(DEVANAGARI.findall(chars)) / len(chars) if chars else 0.0
        if n == 0:
            rows.append((0.0, 0.0, 0, devi))
            continue
        langs = [_lang_of(x) for x in toks]
        hi = sum(1 for l in langs if l == "hi")
        hindi_ratio = hi / n
        minority = min(hi, n - hi) / n
        switches = sum(1 for a, b in zip(langs, langs[1:]) if a != b)
        rows.append((hindi_ratio, minority, switches, devi))
    return pd.DataFrame(rows, columns=["hindi_token_ratio", "mixing_index",
                                       "word_switch_points", "devanagari_ratio"])

In [ ]:
### Metric + tensor helpers (verbatim definitions)

def compute_metrics(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="macro", zero_division=0)
    return {"macro_f1": f, "macro_precision": p, "macro_recall": r,
            "mcc": matthews_corrcoef(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(y_true, y_pred)}

class TextDataset(TorchDataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts, self.labels = list(texts), list(labels)
        self.tok, self.max_len = tokenizer, max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, i):
        enc = self.tok(self.texts[i], truncation=True, max_length=self.max_len,
                       padding="max_length", return_tensors="pt")
        item = {k: v.squeeze(0) for k, v in enc.items()}
        item["labels"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

def compute_metrics_transformers(eval_pred):
    logits, labels = eval_pred
    return compute_metrics(labels, np.argmax(logits, axis=-1))

In [ ]:
# ---- Handoff from 01_data_and_preprocessing.ipynb ---------------------------
# Reconstruct the exact processed `prism` splits the monolith held in memory.
prism = {}
for _k in ("train", "val", "test"):
    _p = RESULTS_DIR / f"prism_{_k}.csv"
    assert _p.exists(), f"Missing {_p} - run 01_data_and_preprocessing.ipynb first"
    _df = pd.read_csv(_p)
    _df["text"] = _df["text"].fillna("").astype(str)
    _df["label"] = _df["label"].astype(int)
    prism[_k] = _df
print("Loaded processed splits:", {k: len(v) for k, v in prism.items()})

In [ ]:
# ---- Handoff from 02 / 03 / 04 -----------------------------------------------
# Reconstruct the exact in-memory state the monolith had after all 4 models ran.
ALL_SEED_RESULTS, PER_CLASS_RESULTS, COSTS = [], [], []
THRESH_CURVES, LOG_HISTORY = {}, {}
for _tag in ("02_b0", "03_hing", "04_controls"):
    _p = STATE_DIR / f"{_tag}_state.json"
    assert _p.exists(), f"Missing {_p} - run notebooks 02, 03, 04 first"
    _d = json.loads(_p.read_text())
    ALL_SEED_RESULTS += _d["all_seed"]
    PER_CLASS_RESULTS += _d["per_class"]
    COSTS += _d["costs"]
    for _k, _v in _d["thresh_curves"].items():
        _mk, _sd = _k.split("|")
        THRESH_CURVES[(_mk, int(_sd))] = (np.array(_v["ths"]), np.array(_v["curve"]))
    for _k, _v in _d["log_history"].items():
        _mk, _sd = _k.split("|")
        LOG_HISTORY[(_mk, int(_sd))] = _v
PRISM_SHA = json.loads((STATE_DIR / "run_config.json").read_text())["prism_sha"]

X_train, y_train = prism["train"]["text"].to_numpy(), prism["train"]["label"].to_numpy()
X_val,   y_val   = prism["val"]["text"].to_numpy(),   prism["val"]["label"].to_numpy()
X_test,  y_test  = prism["test"]["text"].to_numpy(),  prism["test"]["label"].to_numpy()
print("Reconstructed state:", len(ALL_SEED_RESULTS), "result rows,",
      len(PER_CLASS_RESULTS), "per-class rows,", len(COSTS), "cost rows,",
      len(THRESH_CURVES), "threshold curves,", len(LOG_HISTORY), "training logs")

## 5. Evaluation & Statistical Testing

### 5.1 Per-seed results → `results.csv`

In [ ]:
results_df = pd.DataFrame(ALL_SEED_RESULTS)
results_df.to_csv(RESULTS_DIR / "results.csv", index=False)
pd.DataFrame(PER_CLASS_RESULTS).to_csv(RESULTS_DIR / "per_class.csv", index=False)
results_df

### 5.2 Mean summary + bootstrap CI **around the 3-seed mean**

v1 plotted a seed-42 CI around a 3-seed mean. v2 resamples the test set and averages
macro-F1 across seeds *within* each resample, so the interval surrounds the plotted mean.

In [ ]:
def bootstrap_ci_mean(y_true, preds_list, n_boot=BOOTSTRAP_N, seed=0):
    rng = np.random.default_rng(seed)
    idx = np.arange(len(y_true))
    P = np.stack(preds_list)
    scores = np.empty(n_boot)
    for b_i in range(n_boot):
        b = rng.choice(idx, size=len(idx), replace=True)
        scores[b_i] = np.mean([f1_score(y_true[b], P[j][b], average="macro", zero_division=0)
                               for j in range(P.shape[0])])
    return np.percentile(scores, [2.5, 97.5])

summary_rows = []
for key, label in [("b0", "B0 TF-IDF+SVM"), ("hing-roberta", "A0 HingRoBERTa"),
                   ("muril", "A0 MuRIL"), ("xlm-roberta", "A0 XLM-R")]:
    test_rows = [r for r in ALL_SEED_RESULTS if r["key"] == key and r["split"] == "test"]
    if not test_rows:
        continue
    try:
        preds = [np.load(PREDS_DIR / f"{key}_seed{s}_test.npy") for s in SEEDS]
        lo, hi = bootstrap_ci_mean(y_test, preds)
    except FileNotFoundError:
        lo = hi = np.nan
    summary_rows.append({
        "model": label, "key": key,
        "macro_f1_mean": np.mean([r["macro_f1"] for r in test_rows]),
        "macro_f1_std":  np.std([r["macro_f1"] for r in test_rows]),
        "mcc_mean":  np.mean([r["mcc"] for r in test_rows]),
        "balanced_acc_mean": np.mean([r["balanced_accuracy"] for r in test_rows]),
        "threshold_mean": np.nanmean([r.get("threshold", np.nan) for r in test_rows]),
        "ci_low": lo, "ci_high": hi,
    })

summary = pd.DataFrame(summary_rows)
summary.to_csv(RESULTS_DIR / "summary.csv", index=False)
summary.round(4)

### 5.3 McNemar (seed 42) + Holm-corrected significance across 3 comparisons

In [ ]:
def mcnemar(y_true, pred_a, pred_b):
    # Exact binomial McNemar on discordant pairs.
    a_correct, b_correct = pred_a == y_true, pred_b == y_true
    n01 = int(np.sum(a_correct & ~b_correct))   # A right, B wrong
    n10 = int(np.sum(~a_correct & b_correct))   # A wrong, B right
    if n01 + n10 == 0:
        return 1.0, n01, n10
    return stats.binomtest(min(n01, n10), n01 + n10, 0.5).pvalue, n01, n10

b0_ref = np.load(PREDS_DIR / f"b0_seed{SEEDS[0]}_test.npy")
stat_rows = []
for key, label in [("hing-roberta", "A0 HingRoBERTa"), ("muril", "A0 MuRIL"),
                   ("xlm-roberta", "A0 XLM-R")]:
    try:
        a_pred = np.load(PREDS_DIR / f"{key}_seed{SEEDS[0]}_test.npy")
    except FileNotFoundError:
        continue
    p, n01, n10 = mcnemar(y_test, a_pred, b0_ref)
    stat_rows.append({"model": label, "p_value": p, "A_right_B_wrong": n01,
                      "A_wrong_B_right": n10, "p_holm_adj": min(1.0, p * len(SEEDS)),
                      "significant_at_0.05_holm": min(1.0, p * 3) < 0.05})
stats_df = pd.DataFrame(stat_rows)
stats_df.to_csv(RESULTS_DIR / "mcnemar.csv", index=False)
stats_df

### 5.4 Figure 1 — Macro-F1: 3-seed mean with bootstrap CI of the mean

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
colors = {"b0": "#4c8bf5", "hing-roberta": "#2fbf71", "muril": "#f5a623", "xlm-roberta": "#8a63d2"}
for _, row in summary.iterrows():
    ax.errorbar(row["model"], row["macro_f1_mean"],
                yerr=[[max(0.0, row["macro_f1_mean"] - row["ci_low"])],
                      [max(0.0, row["ci_high"] - row["macro_f1_mean"])]],
                fmt="o", capsize=5, color=colors.get(row["key"], "gray"), markersize=9)
    ax.text(row["model"], row["ci_high"] + 0.008, f"{row['macro_f1_mean']:.3f}", ha="center", fontsize=9)
ax.set_ylabel("Test Macro-F1")
ax.set_title(f"Test Macro-F1 — mean over seeds {SEEDS}, bootstrap 95% CI of the mean")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig1_macro_f1_ci.png"); plt.show()

### 5.5 Figure — per-class precision / recall / F1 (the brief's missing view)

In [ ]:
per_class_df = pd.DataFrame(PER_CLASS_RESULTS)
pc_test = per_class_df[per_class_df["split"] == "test"].groupby(["model", "class"])[["precision", "recall", "f1"]].mean().reset_index()
order = ["B0 TF-IDF+SVM", "A0 HingRoBERTa", "A0 MuRIL", "A0 XLM-R"]
pc_test["model"] = pd.Categorical(pc_test["model"], order, ordered=True)
pc_test = pc_test.sort_values(["model", "class"])

fig, ax = plt.subplots(figsize=(11, 4.5))
x = np.arange(len(order))
w = 0.13
for i, metric in enumerate(["precision", "recall", "f1"]):
    for j, cls in enumerate(["non-hate", "hate"]):
        vals = [pc_test[(pc_test["model"] == m) & (pc_test["class"] == cls)][metric].mean() for m in order]
        ax.bar(x + (i - 1) * w + (j - 0.5) * w * 0.9, vals, width=w * 0.85,
               label=f"{metric} · {cls}",
               color=plt.cm.tab10(i), alpha=0.45 + 0.55 * j)
ax.set_xticks(x, ["B0", "HingRoBERTa", "MuRIL", "XLM-R"])
ax.set_ylim(0, 1.05); ax.set_ylabel("score")
ax.set_title("Per-class precision / recall / F1 on test (mean over seeds)")
ax.legend(ncol=3, fontsize=8); ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_per_class.png"); plt.show()
pc_test.round(3)

### 5.6 Figure — cost vs quality (parameters, training time, ms/inference)

In [ ]:
costs_df = pd.DataFrame(COSTS)
costs_df.to_csv(RESULTS_DIR / "costs.csv", index=False)
cost_agg = costs_df.groupby("key").agg(params_m=("params_m", "mean"),
                                       train_seconds=("train_seconds", "mean"),
                                       ms_per_item=("ms_per_item", "mean")).reset_index()
cost_agg = cost_agg.merge(summary[["key", "macro_f1_mean"]], on="key")

fig, ax = plt.subplots(figsize=(7.5, 4.5))
for _, r in cost_agg.iterrows():
    ax.scatter(r["params_m"], r["macro_f1_mean"], s=max(40, r["ms_per_item"] * 8),
               color=colors.get(r["key"], "gray"), alpha=0.8, edgecolor="k")
    ax.annotate(r["key"], (r["params_m"], r["macro_f1_mean"]),
                textcoords="offset points", xytext=(8, 4), fontsize=9)
ax.set_xscale("log")
ax.set_xlabel("parameters (millions, log scale)"); ax.set_ylabel("test macro-F1 (mean)")
ax.set_title("Cost vs quality — bubble size = inference ms/item")
ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_cost_f1.png"); plt.show()
cost_agg.round(3)

### 5.7 Figure — validation threshold sweep (why 0.5 was not blindly used)

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4))
for key in ("hing-roberta", "muril", "xlm-roberta"):
    if (key, SEEDS[0]) in THRESH_CURVES:
        ths, curve = THRESH_CURVES[(key, SEEDS[0])]
        ax.plot(ths, curve, label=key, color=colors.get(key))
        thr_used = [r["threshold"] for r in ALL_SEED_RESULTS
                    if r["key"] == key and r["seed"] == SEEDS[0] and r["split"] == "test"][0]
        ax.axvline(thr_used, color=colors.get(key), ls="--", alpha=0.5)
ax.set_xlabel("decision threshold"); ax.set_ylabel("validation macro-F1")
ax.set_title("Threshold tuned on validation only (dashed = chosen)")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_threshold.png"); plt.show()

for key in ("hing-roberta", "muril", "xlm-roberta"):
    rows = [r for r in ALL_SEED_RESULTS if r["key"] == key and r["split"] == "test" and r["seed"] == SEEDS[0]]
    if rows:
        print(f"{key:12s} tuned thr={rows[0]['threshold']:.2f}  "
              f"tuned-F1={rows[0]['macro_f1']:.4f}  argmax-F1={rows[0]['test_macro_f1_argmax']:.4f}")

### 5.8 Figure — learning curves (HingRoBERTa, seed 42)

In [ ]:
hist = LOG_HISTORY.get(("hing-roberta", SEEDS[0]), [])
if hist:
    steps  = [h["step"] for h in hist if "loss" in h]
    losses = [h["loss"] for h in hist if "loss" in h]
    ev_steps  = [h["step"] for h in hist if "eval_macro_f1" in h]
    ev_f1     = [h["eval_macro_f1"] for h in hist if "eval_macro_f1" in h]
    fig, ax1 = plt.subplots(figsize=(7.5, 4))
    ax1.plot(steps, losses, color="#4c8bf5", label="train loss")
    ax1.set_xlabel("step"); ax1.set_ylabel("loss", color="#4c8bf5")
    ax2 = ax1.twinx()
    ax2.plot(ev_steps, ev_f1, color="#2fbf71", marker="o", label="val macro-F1")
    ax2.set_ylabel("val macro-F1", color="#2fbf71")
    ax1.set_title(f"HingRoBERTa learning curve (seed {SEEDS[0]})")
    fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_learning_curve.png"); plt.show()
else:
    print("No log history (smoke mode?) — skipped.")

## 6. Error Analysis of Code-Mixing Challenges

### 6.1 Best A0 selected on **validation** + confusion matrices

In [ ]:
# v1 selected the backbone on TEST macro-F1. v2 selects on validation; test is reported only.
val_means = {k: np.mean([r["macro_f1"] for r in ALL_SEED_RESULTS
                         if r["key"] == k and r["split"] == "val"])
             for k in ("hing-roberta", "muril", "xlm-roberta")
             if any(r["key"] == k and r["split"] == "val" for r in ALL_SEED_RESULTS)}
best_a0 = max(val_means, key=val_means.get)
print("Val mean macro-F1:", {k: round(v, 4) for k, v in val_means.items()})
print("Best A0 backbone (chosen on VALIDATION):", best_a0)

a0_pred = np.load(PREDS_DIR / f"{best_a0}_seed{SEEDS[0]}_test.npy")
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (name, pred) in zip(axes, [("B0 TF-IDF+SVM", b0_ref), (f"A0 {best_a0}", a0_pred)]):
    cm = confusion_matrix(y_test, pred)
    ax.imshow(cm, cmap="Blues")
    for i in range(2):
        for j in range(2):
            ax.text(j, i, cm[i, j], ha="center", va="center",
                    color="white" if cm[i, j] > cm.max() / 2 else "black")
    ax.set_xticks([0, 1], ["non-hate", "hate"]); ax.set_yticks([0, 1], ["non-hate", "hate"])
    ax.set_xlabel("predicted"); ax.set_ylabel("true"); ax.set_title(name)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_confusion.png"); plt.show()

### 6.2 Performance per language — where does code-mixing hurt?

In [ ]:
test_df = prism["test"].copy()
test_df["b0_pred"] = b0_ref
test_df["a0_pred"] = a0_pred

per_lang = []
for lang, grp in test_df.groupby("lang"):
    per_lang.append({"lang": lang, "n": len(grp),
                     "b0_macro_f1": f1_score(grp["label"], grp["b0_pred"], average="macro", zero_division=0),
                     "a0_macro_f1": f1_score(grp["label"], grp["a0_pred"], average="macro", zero_division=0)})
per_lang = pd.DataFrame(per_lang)
per_lang.to_csv(RESULTS_DIR / "per_language.csv", index=False)
per_lang.round(4)

### 6.3 Misclassifications by **language-mixing** bucket (v2 metric)

Buckets on `mixing_index` (minority-token share), so Romanised Hinglish is finally visible.

In [ ]:
test_mix = code_mixing_stats(test_df["text"])
test_df["mixing_index"] = test_mix["mixing_index"].to_numpy()
test_df["word_switch_points"] = test_mix["word_switch_points"].to_numpy()
test_df["devanagari_ratio"] = test_mix["devanagari_ratio"].to_numpy()
test_df["a0_wrong"] = test_df["a0_pred"] != test_df["label"]

buckets = {
    "monolingual (<0.05)":   test_df["mixing_index"] < 0.05,
    "low mixing (0.05-0.2)": (test_df["mixing_index"] >= 0.05) & (test_df["mixing_index"] < 0.2),
    "code-mixed (0.2-0.5)":  (test_df["mixing_index"] >= 0.2) & (test_df["mixing_index"] < 0.5),
    "highly mixed (>=0.5)":  test_df["mixing_index"] >= 0.5,
}
phen_rows = []
for name, mask in buckets.items():
    sub = test_df[mask]
    if len(sub) == 0:
        continue
    phen_rows.append({"phenomenon": name, "n": len(sub),
                      "a0_error_rate": sub["a0_wrong"].mean(),
                      "a0_macro_f1": f1_score(sub["label"], sub["a0_pred"], average="macro", zero_division=0),
                      "mean_word_switches": sub["word_switch_points"].mean()})
phenomena = pd.DataFrame(phen_rows)
phenomena.to_csv(RESULTS_DIR / "error_phenomena.csv", index=False)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(phenomena["phenomenon"], phenomena["a0_error_rate"], color="#e5484d", alpha=0.8)
for i, (n, e) in enumerate(zip(phenomena["n"], phenomena["a0_error_rate"])):
    ax.text(i, e + 0.005, f"n={n}", ha="center", fontsize=9)
ax.set_ylabel("A0 error rate"); ax.set_ylim(0, max(phenomena["a0_error_rate"]) * 1.25)
ax.set_title(f"A0 ({best_a0}) error rate by language-mixing bucket")
ax.grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_error_phenomena.png"); plt.show()
phenomena.round(4)

### 6.4 Qualitative export — disagreeing examples for the report

In [ ]:
disagree = test_df.loc[test_df["b0_pred"] != test_df["a0_pred"],
                       ["text", "label", "lang", "b0_pred", "a0_pred", "mixing_index", "word_switch_points"]]
disagree = disagree.rename(columns={"label": "true", "b0_pred": "b0", "a0_pred": f"a0_{best_a0}",
                                    "word_switch_points": "code_switch_points"})
disagree.to_csv(RESULTS_DIR / "qualitative_disagreements.csv", index=False)
print(f"Exported {len(disagree)} B0-vs-A0 disagreement examples.")
disagree.head(10)

## 7. Cross-Lingual Transfer — Marathi-English (MeHate)

### 7.1 Load MeHate at a pinned commit (zero-shot only — no training, no leakage)

*Disclosure:* this uses the **small** code-mixed test split (n≈416), not the full ~2.7K
corpus; the sentiment/profanity columns are not used. Both facts are stated in the report.

In [ ]:
def github_head_sha(repo: str) -> str:
    with urllib.request.urlopen(f"https://api.github.com/repos/{repo}/commits/main", timeout=30) as r:
        return json.load(r)["sha"]

MEHATE_SHA = github_head_sha(MEHATE_REPO)
MEHATE_URL = f"https://raw.githubusercontent.com/{MEHATE_REPO}/{MEHATE_SHA}/{MEHATE_CSV}"
print(f"MeHate revision (pinned): {MEHATE_SHA}")

mehate_test = pd.read_csv(MEHATE_URL)
mehate_test = mehate_test.rename(columns={"tweet": "text", "hate": "label"})
mehate_test = mehate_test[["text", "label"]].copy()
mehate_test["text"] = mehate_test["text"].astype(str).map(deidentify)
mehate_test["label"] = mehate_test["label"].astype(float).astype(int)
assert set(mehate_test["label"].unique()) <= {0, 1}, "unexpected MeHate labels"
if SMOKE_TEST:
    mehate_test = mehate_test.head(100).reset_index(drop=True)
print(f"MeHate test: {len(mehate_test)} rows | balance "
      f"{mehate_test['label'].value_counts(normalize=True).round(3).to_dict()}")

In [ ]:
# ---- Handoff: rebuild the trained best-A0 model for section 7.2 --------------
# (the monolith held this in TRAINED; notebooks 03/04 persist the seed-42 weights)
_best_model = AutoModelForSequenceClassification.from_pretrained(MODELS[best_a0], num_labels=2)
_best_model.load_state_dict(torch.load(STATE_DIR / f"model_{best_a0}_seed{SEEDS[0]}.pt",
                                       map_location=DEVICE))
_best_model.to(DEVICE)
TRAINED[(best_a0, SEEDS[0])] = {
    "tokenizer": AutoTokenizer.from_pretrained(MODELS[best_a0]),
    "model": _best_model,
    "val_pred": np.load(PREDS_DIR / f"{best_a0}_seed{SEEDS[0]}_val.npy"),
    "test_pred": np.load(PREDS_DIR / f"{best_a0}_seed{SEEDS[0]}_test.npy"),
    "test_probs": np.load(PREDS_DIR / f"{best_a0}_seed{SEEDS[0]}_test_probs.npy"),
    "threshold": float(np.load(PREDS_DIR / f"{best_a0}_seed{SEEDS[0]}_thr.npy")[0]),
}
print("Restored trained model for zero-shot transfer:", best_a0)

### 7.2 Zero-shot transfer with the validation-selected A0 model (seed 42) + bootstrap CI

In [ ]:
best_trained = TRAINED[(best_a0, SEEDS[0])]
tok_best, model_best = best_trained["tokenizer"], best_trained["model"]

mehate_ds = TextDataset(mehate_test["text"], mehate_test["label"], tok_best, MAX_LEN)
transfer_trainer = Trainer(model=model_best, args=TrainingArguments(
    output_dir=str(WORK_DIR / "ckpt_transfer"),
    per_device_eval_batch_size=BATCH_SIZE * 2, report_to="none"), processing_class=tok_best)
transfer_logits = transfer_trainer.predict(mehate_ds).predictions
mehate_probs = torch.softmax(torch.tensor(transfer_logits), dim=-1)[:, 1].numpy()
mehate_pred = (mehate_probs >= best_trained["threshold"]).astype(int)   # same val-tuned threshold

y_mh = mehate_test["label"].to_numpy()
transfer_f1 = f1_score(y_mh, mehate_pred, average="macro", zero_division=0)
rng = np.random.default_rng(0)
mh_idx = np.arange(len(y_mh))
mh_scores = [f1_score(y_mh[b], mehate_pred[b], average="macro", zero_division=0)
             for b in (rng.choice(mh_idx, len(mh_idx), replace=True) for _ in range(200))]
mh_lo, mh_hi = np.percentile(mh_scores, [2.5, 97.5])
np.save(PREDS_DIR / f"{best_a0}_mehate_seed{SEEDS[0]}.npy", mehate_pred)
print(f"MeHate zero-shot macro-F1 = {transfer_f1:.4f} (95% CI {mh_lo:.4f}–{mh_hi:.4f}, "
      f"n={len(y_mh)}, threshold {best_trained['threshold']:.2f} carried from PRISM-val)")

### 7.3 Transfer table + figure — PRISM (Hindi-English) → MeHate (Marathi-English)

In [ ]:
prism_f1 = float(summary.loc[summary["key"] == best_a0, "macro_f1_mean"].iloc[0])
transfer = pd.DataFrame([{
    "model": f"A0 {best_a0} (trained on PRISM)",
    "PRISM test Macro-F1 (Hindi-English)": prism_f1,
    "MeHate zero-shot Macro-F1 (Marathi-English)": transfer_f1,
    "MeHate CI low": mh_lo, "MeHate CI high": mh_hi,
    "MeHate n": len(y_mh),
    "drop": prism_f1 - transfer_f1,
}])
transfer.to_csv(RESULTS_DIR / "transfer.csv", index=False)

fig, ax = plt.subplots(figsize=(6.5, 4))
vals = [prism_f1, transfer_f1]
ax.bar(["PRISM test\n(Hindi-English)", "MeHate zero-shot\n(Marathi-English)"], vals,
       color=["#2fbf71", "#f5a623"], alpha=0.85)
ax.errorbar(1, transfer_f1, yerr=[[transfer_f1 - mh_lo], [mh_hi - transfer_f1]],
            fmt="none", ecolor="k", capsize=5)
for i, v in enumerate(vals):
    ax.text(i, v + 0.01, f"{v:.3f}", ha="center")
ax.set_ylim(0, 1); ax.set_ylabel("macro-F1")
ax.set_title(f"Zero-shot cross-lingual transfer ({best_a0}, seed {SEEDS[0]})")
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig_transfer.png"); plt.show()
transfer.round(4)

## 8. Self-tests — helpers verified before trusting the results

Fast assertions on the deterministic helpers (mirrors the external pressure test).
If any of these fail, do not trust the sections above.

In [ ]:
checks = []
def expect(name, cond):
    checks.append((name, bool(cond)))
    print(("  PASS  " if cond else "  FAIL  ") + name)

# de-identification
expect("handle -> <USER>", deidentify("hi @rahul!") == "hi <USER>!")
expect("url -> <URL>", deidentify("see www.foo.bar/x ok") == "see <URL> ok")
expect("email fully removed", deidentify("mail anish.date@gmail.com now") == "mail <EMAIL> now")
expect("phone removed", "<PHONE>" in deidentify("call +91-98765-43210 now"))
expect("elongation letters-only", deidentify("sooooo funny 7777777") == "soo funny 7777777")
expect("idempotent", deidentify(deidentify("a @b http://c.d sooooo")) == deidentify("a @b http://c.d sooooo"))

# code-mixing metric: romanised Hinglish must now be VISIBLE
r_hing = code_mixing_stats(["yeh kya bakwaas hai yaar tune kya kiya"]).iloc[0]
expect("romanised Hinglish mixing_index > 0.2 (was 0.0 in v1)", r_hing["mixing_index"] > 0.2)
expect("romanised Hinglish switch points > 0", r_hing["word_switch_points"] > 0)
r_pure = code_mixing_stats(["बहुत बुरा! सच में? नहीं."]).iloc[0]
expect("pure Hindi: no phantom switches from punctuation", r_pure["word_switch_points"] == 0)
r_mono = code_mixing_stats(["this is plain english text only"]).iloc[0]
expect("plain English -> monolingual", r_mono["mixing_index"] < 0.05)

# McNemar matches the exact binomial reference
y12 = np.array([1] * 10 + [0] * 2)
p_a, p_b = np.ones(12, dtype=int), np.array([0] * 10 + [0] * 2)
p_nb, nb01, nb10 = mcnemar(y12, p_a, p_b)
ref = min(1.0, 2 * sum(math.comb(12, i) for i in range(0, 2 + 1)) / 2 ** 12)
expect("McNemar p matches exact reference (10 vs 2)", abs(p_nb - ref) < 1e-9)

# split aliasing
expect("val -> validation alias", pick_split(["train", "validation", "test"], "val") == "validation")

n_fail = sum(1 for _, ok in checks if not ok)
print(f"\nSELF-TESTS: {len(checks)} run, {n_fail} failed")
assert n_fail == 0, "Self-tests failed — do not trust the results above. Fix and re-run."
print("All self-tests passed — helper behaviour verified.")

## 9. Wrap-up — every number traces to `/kaggle/working/results`

In [ ]:
# Environment snapshot for reproducibility
import subprocess
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
(RESULTS_DIR / "requirements-freeze.txt").write_text(freeze)

(RESULTS_DIR / "run_manifest.json").write_text(json.dumps({
    "prism_id": PRISM_ID, "prism_sha": PRISM_SHA,
    "mehate_repo": MEHATE_REPO, "mehate_sha": MEHATE_SHA, "mehate_csv": MEHATE_CSV,
    "models": MODELS, "seeds": SEEDS, "smoke_test": SMOKE_TEST,
    "max_len": MAX_LEN, "epochs": NUM_EPOCHS, "batch_size": BATCH_SIZE,
    "bootstrap_n": BOOTSTRAP_N, "threshold": "tuned on validation, 0.05-0.95 sweep",
}, indent=2))

artifacts = sorted(p.name for p in RESULTS_DIR.iterdir()) + \
            sorted(p.name for p in FIGURES_DIR.iterdir()) + \
            sorted(p.name for p in PREDS_DIR.iterdir())
print("Artifacts for the GitHub commit:")
for a in artifacts:
    print("  •", a)
print("\nDone. Download /kaggle/working before committing.")